In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, input_file_name

spark = (
    SparkSession.builder
    .appName("Medallion Bronze")
    .master("local[*]")
    .getOrCreate()
)

print("========== SPARK APPLICATION INFO ==========")

print("Application ID   :", spark.sparkContext.applicationId)
print("Application Name :", spark.sparkContext.appName)
print("Spark Version    :", spark.version)
print("Master           :", spark.sparkContext.master)
print("Web UI           :", spark.sparkContext.uiWebUrl)
print("Default Parallel :", spark.sparkContext.defaultParallelism)
print("Default Partitons :",spark.conf.get("spark.sql.shuffle.partitions"))

print("============================================")
spark

========== SPARK APPLICATION INFO ==========
Application ID   : local-1792929306613
Application Name : Medallion Bronze
Spark Version    : 3.5.0
Master           : local[*]
Web UI           : http://Arjun:4041
Default Parallel : 18
Default Partitons : 200


In [2]:
from pyspark.sql.types import *

schema = StructType([
    StructField("row_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("sales", StringType(), True),
    StructField("category", StringType(), True),
    StructField("sub_category", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("payment_type", StringType(), True)
])

bronze_df = (
    spark.read
    .option("header", "false")
    .schema(schema)
    .csv("C:/Users/91790/Desktop/sample_data/Midhun_DataSet/txs.csv")
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", input_file_name())
)

bronze_df.show(5)

+------+----------+--------+------+------------------+--------------------+-----------+----------+------------+--------------------+--------------------+
|row_id|order_date|order_id| sales|          category|        sub_category|       city|     state|payment_type|      ingestion_time|         source_file|
+------+----------+--------+------+------------------+--------------------+-----------+----------+------------+--------------------+--------------------+
|     0|06-26-2011| 4007024| 40.33|              NULL|Cardio Machine Ac...|Clarksville| Tennessee|      credit|2026-10-25 17:25:...|file:///C:/Users/...|
|     1|05-26-2011| 4006742|198.44|Exercise & Fitness|Weightlifting Gloves| Long Beach|California|      credit|2026-10-25 17:25:...|file:///C:/Users/...|
|     2|06-01-2011| 4009775|  5.58|Exercise & Fitness|Weightlifting Mac...|    Anaheim|California|      credit|2026-10-25 17:25:...|file:///C:/Users/...|
|     3|06-05-2011| 4002199|198.19|        Gymnastics|    Gymnastics Rings| 

In [4]:
bronze_df.write \
    .mode("overwrite") \
    .parquet("D:/pyspark_vs_code/medallion_project_duckdb/data/bronze")